# W07 - Multiple Regression: Complete Guide

## 1. Introduction: From Simple to Multiple Regression

In the previous modules, we studied Simple Linear Regression, where a single predictor variable (X) was used to explain or predict a response variable (Y). The model had the form:

Y = beta_0 + beta_1 * X + epsilon

This framework is useful, but it is often too simplistic for real-world systems. Most phenomena are influenced by multiple interacting factors simultaneously.

Real-world outcomes rarely depend on only one variable. For example:
| Outcome | Influencing Variables |
|---|---|
| House Price | Size, location, age, number of rooms |
| Salary | Education, experience, industry |
| Exam Score | Study time, sleep, attendance |
| Disease Risk| Genetics, age, lifestyle |

In [ ]:
# Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from mpl_toolkits.mplot3d import Axes3D

# Set display options for better readability
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for complete reproducibility
np.random.seed(42)

print("Libraries successfully imported and environment configured.")

## 2. Data Creation: Synthetic House Pricing Dataset

To demonstrate Multiple Regression concepts, we need data. We will create a self-contained dataset representing house prices. 
We will intentionally build the `Price` variable as a linear combination of several factors:
- Size (Square footage)
- Age (Years since built)
- Location Score (1 to 10 scale)
- Plus some random noise (epsilon)

This perfectly mimics the underlying assumption of Multiple Linear Regression.

In [ ]:
# 1. Generate Synthetic Features
n_samples = 300

# Size: roughly between 800 and 3500 sqft
size_sqft = np.random.normal(2000, 500, n_samples)

# Age: roughly between 0 and 50 years
age_years = np.random.uniform(0, 50, n_samples)

# Location Score: 1 to 10 scale. Let's make it slightly correlated with Size
location_score = np.clip((size_sqft / 1000) + np.random.normal(5, 1.5, n_samples), 1, 10)

# 2. Define the True Mathematical Relationship (The True Betas)
true_beta_0 = 50.0       # Base price in $1000s
true_beta_size = 0.15    # $150 per sqft
true_beta_age = -2.0     # Loses $2000 per year of age
true_beta_loc = 15.0     # Gains $15000 per location score point

# Random Error (epsilon)
epsilon = np.random.normal(0, 30, n_samples) # $30k standard deviation in noise

# 3. Generate the Target Variable (Price in $1000s)
price_k = (true_beta_0 + 
           (true_beta_size * size_sqft) + 
           (true_beta_age * age_years) + 
           (true_beta_loc * location_score) + 
           epsilon)

# Create DataFrame
df_houses = pd.DataFrame({
    'Size_sqft': size_sqft,
    'Age_years': age_years,
    'Location_Score': location_score,
    'Price_k': price_k
})

print("Step 1: Synthetic Housing Dataset Generated.")

In [ ]:
# Let's inspect the first few rows and basic statistics
print("Dataset Preview:")
print(df_houses.head())
print("\nDataset Descriptive Statistics:")
print(df_houses.describe().round(2))

## 3. Core Concept 1: Why Simple Regression Is Often Insufficient

Using only one predictor creates two major problems:
1. **Omitted Variable Bias**: If important predictors are excluded, coefficient estimates become distorted because effects from missing variables leak into included variables.
2. **Limited Explanatory Power**: Single-variable models often leave large amounts of variability unexplained, leading to weak predictions and low R-squared.

Let's see this in action by fitting a Simple Regression model using ONLY Size.

In [ ]:
# Simple Linear Regression: Price ~ Size
X_simple = df_houses['Size_sqft']
X_simple_with_const = sm.add_constant(X_simple) # Adds beta_0 (intercept)
y = df_houses['Price_k']

model_simple = sm.OLS(y, X_simple_with_const).fit()

print("SIMPLE REGRESSION RESULTS:")
print(f"Intercept (beta_0): {model_simple.params['const']:.2f}")
print(f"Size Coefficient (beta_1): {model_simple.params['Size_sqft']:.4f}")
print(f"R-squared: {model_simple.rsquared:.4f}")

# Notice that the estimated Size Coefficient is HIGHER than our true_beta_size (0.15)
# This is Omitted Variable Bias! 'Size' is absorbing the effect of 'Location_Score' 
# because larger houses tend to have slightly better locations in our synthetic data.

### Visualizing the Simple Regression Model
Let's plot how our simple model attempts to explain Price using only Size.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x='Size_sqft', y='Price_k', data=df_houses, 
            scatter_kws={'alpha':0.5, 'color':'blue'}, 
            line_kws={'color':'red', 'linewidth':2})

plt.title('Simple Linear Regression: Price vs Size')
plt.xlabel('House Size (sqft)')
plt.ylabel('Price ($1000s)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The simple model shows a positive trend, but there is massive variance around the line.")
print("This variance represents the missing information (Age, Location).")

## 4. Core Concept 2: The Multiple Regression Model

Multiple Linear Regression extends simple regression by including several predictors simultaneously.

Y = beta_0 + beta_1 * X_1 + beta_2 * X_2 + ... + beta_k * X_k + epsilon

**Key Conceptual Shift:**
Simple regression asks: "How does one variable affect Y?"
Multiple regression asks: "How does each variable affect Y while holding all other variables constant?"

This is called **Partial Effect Interpretation**.

In [ ]:
# Multiple Linear Regression: Price ~ Size + Age + Location
X_multi = df_houses[['Size_sqft', 'Age_years', 'Location_Score']]
X_multi_with_const = sm.add_constant(X_multi)

model_multi = sm.OLS(y, X_multi_with_const).fit()

print(model_multi.summary())

print("\n--- KEY OBSERVATIONS ---")
print(f"Simple Model Size Coef: {model_simple.params['Size_sqft']:.4f}")
print(f"Multi  Model Size Coef: {model_multi.params['Size_sqft']:.4f} (Closer to true value 0.15)")
print(f"Simple Model R-squared: {model_simple.rsquared:.4f}")
print(f"Multi  Model R-squared: {model_multi.rsquared:.4f}")

## 5. Geometry of Multiple Regression

Simple regression fits a line in 2D space.
Multiple regression fits a hyperplane.

If we use two predictors (e.g., Size and Age), the model Y = beta_0 + beta_1 * X_1 + beta_2 * X_2 creates a 2D plane in 3D space.
Let's visualize this 3D hyperplane!

In [ ]:
# Fit a 2-predictor model just for 3D visualization
X_3d = df_houses[['Size_sqft', 'Age_years']]
X_3d_const = sm.add_constant(X_3d)
model_3d = sm.OLS(y, X_3d_const).fit()

# Create a meshgrid for the plane
size_range = np.linspace(df_houses['Size_sqft'].min(), df_houses['Size_sqft'].max(), 20)
age_range = np.linspace(df_houses['Age_years'].min(), df_houses['Age_years'].max(), 20)
xx_size, yy_age = np.meshgrid(size_range, age_range)

# Calculate corresponding Z (Price) values on the plane
zz_price = (model_3d.params['const'] + 
            model_3d.params['Size_sqft'] * xx_size + 
            model_3d.params['Age_years'] * yy_age)

# Plotting
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of actual data points
ax.scatter(df_houses['Size_sqft'], df_houses['Age_years'], df_houses['Price_k'], 
           color='blue', alpha=0.5, label='Actual Data')

# Surface plot of the regression plane
ax.plot_surface(xx_size, yy_age, zz_price, color='red', alpha=0.3)

ax.set_xlabel('Size (sqft)')
ax.set_ylabel('Age (years)')
ax.set_zlabel('Price ($1000s)')
ax.set_title('Multiple Regression: Fitting a Hyperplane in 3D Space')

plt.show()

## 6. Core Concept 3: R-squared vs Adjusted R-squared

**R-squared (Coefficient of Determination)** measures the proportion of variability in Y explained collectively by all predictors.

**The Danger of Naive R-squared**:
Adding more variables ALWAYS increases R-squared (or keeps it the same). Even completely useless/random predictors will inflate apparent model quality. This creates massive Overfitting Risk.

**Adjusted R-squared**:
To address this, we use Adjusted R-squared, which penalizes unnecessary complexity. Unlike ordinary R-squared, Adjusted R-squared can actually DECREASE if you add a useless variable.

In [ ]:
# Let's demonstrate the danger of R-squared by adding random 'junk' variables
df_junk = df_houses.copy()

r2_list = []
adj_r2_list = []

# Base model features
features = ['Size_sqft', 'Age_years', 'Location_Score']

for i in range(1, 21): # Add 20 completely random junk variables one by one
    junk_col = f'Junk_Var_{i}'
    df_junk[junk_col] = np.random.normal(0, 1, n_samples)
    features.append(junk_col)
    
    X_temp = sm.add_constant(df_junk[features])
    temp_model = sm.OLS(y, X_temp).fit()
    
    r2_list.append(temp_model.rsquared)
    adj_r2_list.append(temp_model.rsquared_adj)

# Visualization
plt.figure(figsize=(10, 5))
plt.plot(range(1, 21), r2_list, marker='o', label='R-squared (Keeps Increasing)', color='green')
plt.plot(range(1, 21), adj_r2_list, marker='x', label='Adjusted R-squared (Penalizes Junk)', color='red')

plt.title('R-squared vs Adjusted R-squared when adding Useless Predictors')
plt.xlabel('Number of Random Junk Variables Added')
plt.ylabel('Score')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Notice how Standard R-squared slowly creeps up even though the variables contain zero real information.")

## 7. Core Concept 4: Multicollinearity (Interpretation Challenges)

Interpreting multiple regression is much harder than simple regression because predictors may interact statistically.

One major issue is **Multicollinearity**: Predictors correlated with each other. When two predictors carry the exact same information, the model cannot distinguish their individual effects, leading to massive instability in coefficients.

Let's intentionally create a multicollinear feature: Size in Square Meters.

In [ ]:
# Create highly collinear variable: sqft converted to sq_meters
df_houses['Size_sqm'] = df_houses['Size_sqft'] / 10.764

# Check correlation matrix
corr_matrix = df_houses[['Price_k', 'Size_sqft', 'Size_sqm', 'Age_years', 'Location_Score']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix (Notice Size_sqft and Size_sqm correlation = 1.0)')
plt.tight_layout()
plt.show()

In [ ]:
# Fit model with collinear variables
X_collinear = df_houses[['Size_sqft', 'Size_sqm', 'Age_years', 'Location_Score']]
X_collinear_const = sm.add_constant(X_collinear)

model_collinear = sm.OLS(y, X_collinear_const).fit()

print("Model Results WITH Multicollinearity:")
print(model_collinear.summary())

print("\n--- WARNING ---")
print("Statsmodels successfully outputs a warning about strong multicollinearity.")
print("Notice the standard errors for the Size variables might be inflated, or the coefficients behave erratically.")

## 8. Matrix Representation of Multiple Regression

Multiple regression is naturally expressed using linear algebra:
Y = X * beta + epsilon

Where:
- Y is the Response vector
- X is the Design matrix (features plus a column of 1s for the intercept)
- beta is the Coefficient vector
- epsilon is the Error vector

The Least Squares estimate for beta is mathematically solved as:
beta_hat = (X^T * X)^-1 * X^T * Y

Let's implement this manually using Numpy to prove how it works under the hood.

In [ ]:
# Manual Matrix Math Implementation of OLS
# 1. Extract matrices
Y_mat = df_houses['Price_k'].values

# Create design matrix X (Add column of 1s manually)
ones_col = np.ones(n_samples)
X_features = df_houses[['Size_sqft', 'Age_years', 'Location_Score']].values
X_mat = np.column_stack((ones_col, X_features))

# 2. Calculate (X^T * X)
X_T_X = X_mat.T @ X_mat

# 3. Calculate Inverse of (X^T * X)
X_T_X_inv = np.linalg.inv(X_T_X)

# 4. Calculate X^T * Y
X_T_Y = X_mat.T @ Y_mat

# 5. Final Beta calculation: Beta = (X^T * X)^-1 * X^T * Y
beta_hat = X_T_X_inv @ X_T_Y

print("Manual Matrix Math Coefficients:")
print(f"Intercept:      {beta_hat[0]:.4f}")
print(f"Size_sqft:      {beta_hat[1]:.4f}")
print(f"Age_years:      {beta_hat[2]:.4f}")
print(f"Location_Score: {beta_hat[3]:.4f}")

print("\nCompare with Statsmodels Output:")
print(model_multi.params.values.round(4))
print("They match perfectly! Machine Learning models use this exact linear algebra foundation.")

## 9. Model Diagnostics (Residual Analysis)

A critical part of the Multiple Regression workflow is validating assumptions by checking Residuals (Errors).
Residuals = Actual Y - Predicted Y

We need to check:
1. Linearity: Residuals should have no pattern against predicted values.
2. Normality: Residuals should be approximately normally distributed.

In [ ]:
# Get predictions and residuals from our good model
y_pred = model_multi.predict(X_multi_with_const)
residuals = model_multi.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residuals vs Predicted Values
sns.residplot(x=y_pred, y=residuals, lowess=True, 
              scatter_kws={'alpha': 0.5}, line_kws={'color': 'red', 'lw': 2}, ax=axes[0])
axes[0].set_title('Residuals vs Predicted (Check Linearity & Equal Variance)')
axes[0].set_xlabel('Predicted Price ($1000s)')
axes[0].set_ylabel('Residuals')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].grid(alpha=0.3)

# Plot 2: Distribution of Residuals
sns.histplot(residuals, kde=True, ax=axes[1], color='purple')
axes[1].set_title('Distribution of Residuals (Check Normality)')
axes[1].set_xlabel('Residual Error')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Diagnostics look good. Errors are centered around zero and normally distributed.")

## 10. Practice Exercises

### Exercise 1: Building a Sub-Model
Using `df_houses`, build a multiple regression model predicting `Price_k` using ONLY `Age_years` and `Location_Score`. Print the summary and state the Adjusted R-squared.

In [ ]:
# Exercise 1 Setup
print("Target: Build model with Age_years and Location_Score")
# Your code here:

In [ ]:
# Exercise 1 Solution
X_ex1 = df_houses[['Age_years', 'Location_Score']]
X_ex1_const = sm.add_constant(X_ex1)
y_target = df_houses['Price_k']

model_ex1 = sm.OLS(y_target, X_ex1_const).fit()
print(model_ex1.summary())
print(f"\nAdjusted R-squared for Sub-Model: {model_ex1.rsquared_adj:.4f}")

### Exercise 2: Comparing Explanatory Power
Write a short script to compare the Adjusted R-squared of three models:
1. Size Only
2. Age & Location Only
3. Size, Age, & Location
Print which one performs the best.

In [ ]:
# Exercise 2 Setup
print("Target: Compare 3 models' Adjusted R-squared")
# Your code here:

In [ ]:
# Exercise 2 Solution
# Model 1 (Already built as model_simple, but let's redo for cleanliness)
adj_r2_1 = sm.OLS(df_houses['Price_k'], sm.add_constant(df_houses[['Size_sqft']])).fit().rsquared_adj

# Model 2
adj_r2_2 = sm.OLS(df_houses['Price_k'], sm.add_constant(df_houses[['Age_years', 'Location_Score']])).fit().rsquared_adj

# Model 3
adj_r2_3 = sm.OLS(df_houses['Price_k'], sm.add_constant(df_houses[['Size_sqft', 'Age_years', 'Location_Score']])).fit().rsquared_adj

print(f"Model 1 (Size Only) Adj R2:            {adj_r2_1:.4f}")
print(f"Model 2 (Age & Location) Adj R2:       {adj_r2_2:.4f}")
print(f"Model 3 (Size, Age, Location) Adj R2:  {adj_r2_3:.4f}")

best = max(adj_r2_1, adj_r2_2, adj_r2_3)
print(f"\nThe best performing model is Model 3 with Adj R2 = {best:.4f}")

## 11. Summary and Key Takeaways

- **Concept**: Multiple regression models multiple predictors simultaneously.
- **Partial Effects**: Each coefficient measures the effect of a predictor while controlling for (holding constant) all others.
- **Omitted Variable Bias**: Leaving out important factors distorts the effects of the included factors.
- **R-squared vs Adj R-squared**: Standard R-squared always goes up with new variables. Adjusted R-squared penalizes useless variables, offering a more honest evaluation.
- **Linear Algebra**: The underlying math is a beautiful matrix operation: beta_hat = (X^T * X)^-1 * X^T * Y.
- **Real World Application**: This is the gateway from basic statistics into machine learning, predictive modeling, and econometric causal inference.

## 12. Visualization Gallery
A final recap gallery of variable relationships.

In [ ]:
# Pairplot to view all bi-variate relationships simultaneously
sns.pairplot(df_houses[['Price_k', 'Size_sqft', 'Age_years', 'Location_Score']], 
             diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of All Variables in Multiple Regression', y=1.02)
plt.show()

print("Module Complete: W07 - Multiple Regression Successfully Rendered.")